# Call the Team 4 model server

Sends a bedroom count and a floor area to `/search` and prints the 6 plans it returns.

The free Render service **sleeps after 15 minutes** without requests: the first call can take about a minute.

Install once: `pip install gradio_client==2.7.2 requests`

In [1]:
import os, json
# The live server on Render. Set MODEL_URL=http://127.0.0.1:7860 to test a local `python app.py`.
URL = os.environ.get("MODEL_URL", "https://team4-resplan-server.onrender.com")
print("Calling", URL)

Calling https://team4-resplan-server.onrender.com


## 1. With `gradio_client` (easiest)

In [2]:
from gradio_client import Client

client = Client(URL, verbose=False)
result = client.predict(3, 1100, api_name="/search")   # bedrooms, sqft
print("Query:", result["query"], "->", result["count"], "plans")
for p in result["plans"]:
    s = p["scores"]
    print(f'id {p["id"]:>6}  {p["bedrooms"]} bed {p["bathrooms"]} bath  {p["area_sqft"]:>7} sqft  '
          f'daylight {s["daylight_score"]:.2f}  circulation {s["circulation_score"]:.2f}  dark rooms {s["dark_rooms"]}')

Query: {'bedrooms': 3, 'sqft': 1100.0} -> 6 plans
id   8741  3 bed 2 bath   1100.0 sqft  daylight 0.60  circulation 0.67  dark rooms 0
id  11021  3 bed 2 bath   1100.0 sqft  daylight 0.80  circulation 0.75  dark rooms 0
id   6952  3 bed 2 bath   1100.0 sqft  daylight 0.60  circulation 0.67  dark rooms 0
id   5160  3 bed 2 bath   1100.0 sqft  daylight 0.40  circulation 0.70  dark rooms 1
id  10702  3 bed 3 bath   1100.0 sqft  daylight 0.80  circulation 0.75  dark rooms 0
id  11124  3 bed 3 bath   1100.0 sqft  daylight 0.80  circulation 0.60  dark rooms 0


## 2. One plan in full: room counts, relationship matrix, scores

`relation[a][b]` = number of connections between room type `a` and room type `b` (order in `room_types`).

In [3]:
plan = result["plans"][0]
types = result["room_types"]
print("Room counts:", plan["room_counts"])
print("Scores:", plan["scores"])
print()
print(" " * 11 + "".join(f"{t[:6]:>7}" for t in types))
for name, row in zip(types, plan["relation"]):
    print(f"{name:>10} " + "".join(f"{v:>7}" for v in row))

Room counts: {'living': 1, 'bedroom': 3, 'kitchen': 1, 'bathroom': 2, 'balcony': 2, 'storage': 0, 'stair': 0, 'front_door': 1}
Scores: {'circulation_score': 0.667, 'daylight_score': 0.6, 'cross_vent_rooms': 1, 'dark_rooms': 0, 'missing_door': False}

            living bedroo kitche bathro balcon storag  stair front_
    living       0      3      1      2      1      0      0      1
   bedroom       3      1      1      3      1      0      0      0
   kitchen       1      1      0      0      0      0      0      0
  bathroom       2      3      0      0      1      0      0      0
   balcony       1      1      0      1      0      0      0      0
   storage       0      0      0      0      0      0      0      0
     stair       0      0      0      0      0      0      0      0
front_door       1      0      0      0      0      0      0      0


## 3. The same call over plain HTTP (what the Cloudflare app will do)

In [4]:
import requests

r = requests.post(f"{URL}/gradio_api/call/search", json={"data": [2, 800]}, timeout=120)
event_id = r.json()["event_id"]
r = requests.get(f"{URL}/gradio_api/call/search/{event_id}", timeout=120)
line = [l for l in r.text.splitlines() if l.startswith("data:")][-1]
answer = json.loads(line[len("data:"):])[0]
print(answer["query"], [p["id"] for p in answer["plans"]])

{'bedrooms': 2, 'sqft': 800.0} [13325, 3308, 11064, 5233, 6863, 13010]
